# 01 · EDA — SECOM

fab 센서 데이터: 구조 · 결측 · 클래스 불균형 · 상수 센서 · 상관 · 시간 drift.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data import load_secom
X, y, ts = load_secom()
print("X:", X.shape, "| fail rate:", round(float(y.mean()), 4))

## 클래스 불균형 — 불량이 소수(~6.6%). Accuracy가 무의미한 이유.

In [ ]:
vc = y.value_counts().sort_index()
ax = vc.plot(kind="bar", color=["#163C73", "#C0392B"])
ax.set_xticklabels(["pass (0)", "fail (1)"], rotation=0)
plt.title(f"class balance — fail {y.mean():.1%}"); plt.ylabel("wafers"); plt.tight_layout(); plt.show()

## 결측 구조 — 센서별 결측 비율

In [ ]:
miss = X.isna().mean().sort_values(ascending=False)
print("columns >50% missing:", int((miss > 0.5).sum()))
miss.head(30).plot(kind="bar", figsize=(11, 3), color="#1A4D8F")
plt.title("top-30 missing sensors"); plt.ylabel("missing frac"); plt.tight_layout(); plt.show()

## 상수 / 근상수 센서 — 정보 없는 변수 (분산 0)

In [ ]:
nu = X.nunique()
print("constant sensors  :", int((nu <= 1).sum()))
print("near-constant (<=2):", int((nu <= 2).sum()))

## 상관 구조 — 센서 다중공선성(앞 100개)

In [ ]:
sub = X.iloc[:, :100].fillna(X.iloc[:, :100].median())
corr = sub.corr().abs()
sns.heatmap(corr, cmap="viridis", xticklabels=False, yticklabels=False)
plt.title("|corr| of first 100 sensors"); plt.tight_layout(); plt.show()
tri = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
print("highly correlated pairs (>0.95):", int((tri > 0.95).sum().sum()))

## 시간 drift — 주차별 웨이퍼 수 (랜덤 split이 낙관적인 이유)

In [ ]:
ts.dt.to_period("W").value_counts().sort_index().plot(figsize=(11, 3), color="#163C73")
plt.title("wafers per week"); plt.ylabel("count"); plt.tight_layout(); plt.show()